# Fase 3 — Preparação dos Dados
Tratamentos definidos a partir dos defeitos encontrados na Fase 2.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
RAIZ = Path('..')
DADOS_BRUTOS = RAIZ / 'dados' / 'brutos'
ESTADO = RAIZ / 'estado'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

lojas = pd.read_csv(DADOS_BRUTOS / 'loja.csv')

## 1. Imputação de distancia_concorrente

In [2]:
lojas_tratadas = lojas.copy()
mediana_distancia = lojas_tratadas['distancia_concorrente'].median()

# a marcação preserva quais lojas tiveram distância imputada
lojas_tratadas['distancia_imputada'] = lojas_tratadas['distancia_concorrente'].isna().astype(int)
lojas_tratadas['distancia_concorrente'] = lojas_tratadas['distancia_concorrente'].fillna(mediana_distancia)

print(f'mediana usada: {mediana_distancia:,.0f}')
print('lojas imputadas:', lojas_tratadas.loc[lojas_tratadas['distancia_imputada'] == 1, 'loja'].tolist())
print('nulos restantes:', int(lojas_tratadas['distancia_concorrente'].isna().sum()))

mediana usada: 2,325
lojas imputadas: [291, 622, 879]
nulos restantes: 0


## 2. log1p apenas nas variáveis assimétricas
Critério: |skew| > 1. Variáveis binárias ficam fora, pois o log só reescala o 1.

In [3]:
LIMITE_ASSIMETRIA = 1

colunas_numericas = [c for c in lojas_tratadas.select_dtypes('number').columns if c not in ('loja', 'distancia_imputada')]
diagnostico = pd.DataFrame({
    'skew': [lojas_tratadas[c].skew() for c in colunas_numericas],
    'valores_distintos': [lojas_tratadas[c].nunique() for c in colunas_numericas],
}, index=colunas_numericas)
diagnostico['aplicar_log1p'] = (diagnostico['skew'].abs() > LIMITE_ASSIMETRIA) & (diagnostico['valores_distintos'] > 2)
print(diagnostico.round(3).to_string())

colunas_log = diagnostico.index[diagnostico['aplicar_log1p']].tolist()
for coluna in colunas_log:
    lojas_tratadas[f'{coluna}_log'] = np.log1p(lojas_tratadas[coluna])
print('\ncolunas transformadas:', colunas_log)
print('mantidas como estavam:', [c for c in colunas_numericas if c not in colunas_log])

                        skew  valores_distintos  aplicar_log1p
distancia_concorrente  2.934                655           True
promo_continua        -0.049                  2          False

colunas transformadas: ['distancia_concorrente']
mantidas como estavam: ['promo_continua']


In [4]:
for coluna in colunas_log:
    print(f"{coluna}: skew {lojas_tratadas[coluna].skew():.3f} -> {lojas_tratadas[f'{coluna}_log'].skew():.3f}")
print()
print(lojas_tratadas[[f'{c}_log' for c in colunas_log]].describe().round(3).to_string())

distancia_concorrente: skew 2.934 -> -0.352

       distancia_concorrente_log
count                   1115.000
mean                       7.646
std                        1.550
min                        3.045
25%                        6.581
50%                        7.752
75%                        8.836
max                       11.237


In [5]:
lojas_tratadas.to_pickle(ESTADO / 'lojas_tratadas.pkl')
lojas_tratadas.head()

,loja,tipo_loja,sortimento,distancia_concorrente,promo_continua,distancia_imputada,distancia_concorrente_log
0,1,c,a,1270.0,0,0,7.147559
1,2,a,a,570.0,1,0,6.347389
2,3,a,a,14130.0,1,0,9.556126
3,4,c,c,620.0,0,0,6.431331
4,5,a,a,29910.0,0,0,10.305982
